In [4]:
# Data preparation
required_columns = ["claim_status", "vehicle_age", "customer_age", "subscription_length"]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["claim_status"] = pd.to_numeric(df["claim_status"], errors="coerce")
df = df.dropna(subset=["claim_status"]).copy()
df["claim_status"] = df["claim_status"].astype(int)
df = df[df["claim_status"].isin([0, 1])].copy()

for col in ["vehicle_age", "customer_age", "subscription_length", "region_density"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["Claim Label"] = df["claim_status"].map({0: "No Claim", 1: "Claim"})
df["Vehicle Age Group"] = pd.cut(
    df["vehicle_age"], bins=[-np.inf, 1, 3, 5, 10, np.inf],
    labels=["≤1 year", "1–3 years", "3–5 years", "5–10 years", "10+ years"],
    include_lowest=True
)
df["Customer Age Group"] = pd.cut(
    df["customer_age"], bins=[-np.inf, 25, 35, 45, 55, np.inf],
    labels=["≤25", "26–35", "36–45", "46–55", "56+"],
    include_lowest=True
)
df["Subscription Group"] = pd.cut(
    df["subscription_length"], bins=[-np.inf, 1, 3, 5, 8, np.inf],
    labels=["≤1 year", "1–3 years", "3–5 years", "5–8 years", "8+ years"],
    include_lowest=True
)

print("Data preparation completed.")
print("Claim status counts:")
display(df["Claim Label"].value_counts().rename_axis("Status").to_frame("Records"))

Data preparation completed.
Claim status counts:


,Records
Status,
No Claim,54844
Claim,3748


In [1]:
# Interactive dashboard and visualizations
BLUE = "#527DA8"
RED = "#DF5B61"
GREEN = "#6C9A7D"

def options_for(column):
    if column not in df.columns:
        return ["All"]
    vals = sorted(df[column].dropna().astype(str).unique().tolist())
    return ["All"] + vals

fuel_filter = widgets.Dropdown(
    options=options_for("fuel_type"), value="All",
    description="Fuel:", layout=widgets.Layout(width="32%")
)
region_filter = widgets.Dropdown(
    options=options_for("region_code"), value="All",
    description="Region:", layout=widgets.Layout(width="32%")
)
segment_filter = widgets.Dropdown(
    options=options_for("segment"), value="All",
    description="Segment:", layout=widgets.Layout(width="32%")
)
output = widgets.Output()

def make_card(title, value, color):
    return f'''
    <div style="flex:1;min-width:170px;background:#fff;border-top:4px solid {color};
                padding:16px;border-radius:8px;box-shadow:0 2px 7px #ddd;">
      <div style="font-size:14px;color:#555">{title}</div>
      <div style="font-size:30px;margin-top:10px;color:#222;font-weight:600">{value}</div>
    </div>'''

def build_dashboard(*args):
    data = df.copy()
    if fuel_filter.value != "All" and "fuel_type" in data:
        data = data[data["fuel_type"].astype(str) == fuel_filter.value]
    if region_filter.value != "All" and "region_code" in data:
        data = data[data["region_code"].astype(str) == region_filter.value]
    if segment_filter.value != "All" and "segment" in data:
        data = data[data["segment"].astype(str) == segment_filter.value]

    with output:
        output.clear_output(wait=True)
        if data.empty:
            display(HTML("<h3>No records match these filters.</h3>"))
            return

        total = len(data)
        claims = int((data["claim_status"] == 1).sum())
        no_claims = int((data["claim_status"] == 0).sum())
        rate = claims / total * 100 if total else 0

        cards = [
            make_card("Total Records", f"{total:,}", BLUE),
            make_card("Claim Records", f"{claims:,}", RED),
            make_card("Claim Rate", f"{rate:.2f}%", GREEN),
            make_card("No-Claim Records", f"{no_claims:,}", BLUE),
        ]
        display(HTML(
            '<h2 style="font-family:Arial;font-weight:500">Insurance Claim Status Analysis Dashboard</h2>'
            '<div style="display:flex;gap:14px;flex-wrap:wrap">' + "".join(cards) + '</div>'
        ))

        # 1. Claim vs no-claim share
        counts = data["Claim Label"].value_counts().reindex(["Claim", "No Claim"]).dropna()
        fig = go.Figure(go.Pie(
            labels=counts.index, values=counts.values, hole=0.48,
            marker_colors=[RED if x == "Claim" else BLUE for x in counts.index],
            textinfo="label+percent"
        ))
        fig.update_layout(title="1. Claim vs No Claim", template="plotly_white", height=360)
        display(fig)

        # 2. Claim rate by fuel type
        if "fuel_type" in data.columns:
            s = data.groupby("fuel_type", dropna=False).agg(
                Records=("claim_status", "size"), Claim_Rate=("claim_status", "mean")
            ).reset_index()
            s["Claim_Rate"] *= 100
            fig = px.bar(s.sort_values("Claim_Rate"), x="Claim_Rate", y="fuel_type",
                         orientation="h", title="2. Claim Rate by Fuel Type",
                         labels={"Claim_Rate":"Claim Rate (%)","fuel_type":"Fuel Type"},
                         hover_data=["Records"], color_discrete_sequence=[BLUE])
            fig.update_layout(template="plotly_white", height=350)
            display(fig)

        # 3. Claim rate by subscription length
        s = data.dropna(subset=["Subscription Group"]).groupby(
            "Subscription Group", observed=False
        ).agg(Claim_Rate=("claim_status","mean"), Records=("claim_status","size")).reset_index()
        s["Claim_Rate"] *= 100
        fig = px.line(s, x="Subscription Group", y="Claim_Rate", markers=True,
                      title="3. Claim Rate by Subscription Length",
                      labels={"Claim_Rate":"Claim Rate (%)"}, hover_data=["Records"],
                      color_discrete_sequence=[RED])
        fig.update_layout(template="plotly_white", height=350)
        display(fig)

        # 4. Customer age vs vehicle age
        scatter_cols = ["customer_age", "vehicle_age", "Claim Label"]
        s = data[scatter_cols].dropna()
        if len(s) > 5000:
            s = s.sample(5000, random_state=42)
        fig = px.scatter(s, x="vehicle_age", y="customer_age", color="Claim Label",
                         title="4. Customer Age vs Vehicle Age",
                         color_discrete_map={"Claim":RED,"No Claim":BLUE}, opacity=0.55,
                         labels={"vehicle_age":"Vehicle Age (years)","customer_age":"Customer Age (years)"})
        fig.update_layout(template="plotly_white", height=400)
        display(fig)

        # 5. Claim-rate heatmap by age groups
        heat = data.pivot_table(index="Vehicle Age Group", columns="Customer Age Group",
                                values="claim_status", aggfunc="mean", observed=False) * 100
        fig = px.imshow(heat, text_auto=".1f", aspect="auto", color_continuous_scale="Blues",
                        title="5. Claim Rate by Vehicle Age and Customer Age",
                        labels={"x":"Customer Age Group","y":"Vehicle Age Group","color":"Claim Rate (%)"})
        fig.update_layout(template="plotly_white", height=380)
        display(fig)

        # 6. Claim rate by engine type (top 10 by frequency)
        if "engine_type" in data.columns:
            common = data["engine_type"].value_counts().head(10).index
            s = data[data["engine_type"].isin(common)].groupby("engine_type", dropna=False).agg(
                Claim_Rate=("claim_status","mean"), Records=("claim_status","size")
            ).reset_index()
            s["Claim_Rate"] *= 100
            fig = px.bar(s.sort_values("Claim_Rate"), x="Claim_Rate", y="engine_type",
                         orientation="h", title="6. Claim Rate by Engine Type (Top 10)",
                         labels={"Claim_Rate":"Claim Rate (%)","engine_type":"Engine Type"},
                         hover_data=["Records"], color_discrete_sequence=[RED])
            fig.update_layout(template="plotly_white", height=430)
            display(fig)

        # 7. Claim rate by transmission type
        if "transmission_type" in data.columns:
            s = data.groupby("transmission_type", dropna=False).agg(
                Claim_Rate=("claim_status","mean"), Records=("claim_status","size")
            ).reset_index()
            s["Claim_Rate"] *= 100
            fig = px.bar(s, x="transmission_type", y="Claim_Rate",
                         title="7. Claim Rate by Transmission Type",
                         labels={"transmission_type":"Transmission Type","Claim_Rate":"Claim Rate (%)"},
                         hover_data=["Records"], color_discrete_sequence=[BLUE])
            fig.update_layout(template="plotly_white", height=350)
            display(fig)

        # 8. Claim rate by safety features (correct dataset column names)
        safety_columns = {
            "Adjustable Steering": "is_adjustable_steering",
            "Brake Assist": "is_brake_assist",
            "Parking Sensors": "is_parking_sensors",
            "Speed Alert": "is_speed_alert",
        }
        rows = []
        for label, col in safety_columns.items():
            if col in data.columns:
                grouped = data.groupby(col, dropna=False)["claim_status"].agg(["mean","size"]).reset_index()
                for _, row in grouped.iterrows():
                    rows.append({"Feature":label, "Feature Value":str(row[col]),
                                 "Claim Rate (%)":row["mean"]*100, "Records":int(row["size"])})
        if rows:
            s = pd.DataFrame(rows)
            fig = px.bar(s, x="Feature", y="Claim Rate (%)", color="Feature Value", barmode="group",
                         title="8. Claim Rate by Safety Features", hover_data=["Records"])
            fig.update_layout(template="plotly_white", height=400)
            display(fig)

        # 9. Claim rate by region density quintile
        if "region_density" in data.columns and data["region_density"].nunique() > 1:
            s = data.dropna(subset=["region_density"]).copy()
            s["Density Group"] = pd.qcut(s["region_density"], q=5, duplicates="drop")
            s = s.groupby("Density Group", observed=False).agg(
                Claim_Rate=("claim_status","mean"), Records=("claim_status","size")
            ).reset_index()
            s["Claim_Rate"] *= 100
            s["Density Group"] = s["Density Group"].astype(str)
            fig = px.bar(s, x="Density Group", y="Claim_Rate",
                         title="9. Claim Rate by Region Density",
                         labels={"Density Group":"Region Density Band","Claim_Rate":"Claim Rate (%)"},
                         hover_data=["Records"], color_discrete_sequence=[BLUE])
            fig.update_layout(template="plotly_white", height=350)
            display(fig)

        # 10. Customer age distribution by claim status
        fig = px.histogram(data.dropna(subset=["customer_age"]), x="customer_age", color="Claim Label",
                           barmode="overlay", opacity=0.65, nbins=30,
                           title="10. Customer Age Distribution by Claim Status",
                           color_discrete_map={"Claim":RED,"No Claim":BLUE},
                           labels={"customer_age":"Customer Age (years)"})
        fig.update_layout(template="plotly_white", height=400)
        display(fig)

        # Summary table
        summary = data.groupby("Claim Label", dropna=False).agg(
            Records=("claim_status","size"),
            Average_Customer_Age=("customer_age","mean"),
            Average_Vehicle_Age=("vehicle_age","mean"),
            Claim_Rate=("claim_status","mean")
        ).reset_index()
        summary["Claim_Rate"] = (summary["Claim_Rate"] * 100).round(2)
        summary["Average_Customer_Age"] = summary["Average_Customer_Age"].round(1)
        summary["Average_Vehicle_Age"] = summary["Average_Vehicle_Age"].round(2)
        display(HTML("<h3>Summary Table</h3>"))
        display(summary.rename(columns={
            "Claim Label":"Claim Status", "Average_Customer_Age":"Avg Customer Age",
            "Average_Vehicle_Age":"Avg Vehicle Age", "Claim_Rate":"Claim Rate (%)"
        }))
        display(HTML('<p style="color:#666;font-size:12px">Claim rates describe patterns in this dataset. They do not prove causation or claim approval.</p>'))

fuel_filter.observe(build_dashboard, names="value")
region_filter.observe(build_dashboard, names="value")
segment_filter.observe(build_dashboard, names="value")

display(widgets.HBox(
    [fuel_filter, region_filter, segment_filter],
    layout=widgets.Layout(width="100%", flex_flow="row wrap")
))
display(output)
build_dashboard()

NameError: name 'widgets' is not defined